In [15]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_absolute_percentage_error
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
import torch.optim as optim

# ---------------- Load Data ---------------- #
train_df = pd.read_csv('dataset/train.csv')
test_df = pd.read_csv('dataset/test.csv')
test_df = test_df.drop(columns=['ID'])  # Drop ID column for features
sample_submission = pd.read_csv('dataset/sample_solution.csv')

X = train_df.drop([f'BlendProperty{i}' for i in range(1, 11)], axis=1)
y = train_df[[f'BlendProperty{i}' for i in range(1, 11)]]

# ---------------- Preprocessing ---------------- #
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_test_scaled = scaler.transform(test_df)

X_train, X_val, y_train, y_val = train_test_split(X_scaled, y.values, test_size=0.2, random_state=42)

# ---------------- PyTorch Dataset ---------------- #
class FuelDataset(Dataset):
    def __init__(self, X, y=None):
        self.X = torch.tensor(X, dtype=torch.float32)
        self.y = torch.tensor(y, dtype=torch.float32) if y is not None else None

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        if self.y is not None:
            return self.X[idx], self.y[idx]
        else:
            return self.X[idx]

train_dataset = FuelDataset(X_train, y_train)
val_dataset = FuelDataset(X_val, y_val)
test_dataset = FuelDataset(X_test_scaled)

train_loader = DataLoader(train_dataset, batch_size=64, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=64, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=64, shuffle=False)

# ---------------- Neural Network ---------------- #
class BlendNet(nn.Module):
    def __init__(self, input_dim, output_dim):
        super(BlendNet, self).__init__()
        self.model = nn.Sequential(
            nn.Linear(input_dim, 512),
            nn.BatchNorm1d(512),
            nn.ReLU(),
            nn.Linear(512, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(128, output_dim)
        )

    def forward(self, x):
        return self.model(x)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = BlendNet(input_dim=X.shape[1], output_dim=10).to(device)

# ---------------- Training Setup ---------------- #
criterion = nn.L1Loss()  # MAE Loss aligns with MAPE goal
optimizer = optim.Adam(model.parameters(), lr=1e-3)
epochs = 500
patience = 20
best_val_loss = np.inf
early_stop_counter = 0

# ---------------- Training Loop ---------------- #
for epoch in range(epochs):
    model.train()
    train_losses = []
    for X_batch, y_batch in train_loader:
        X_batch, y_batch = X_batch.to(device), y_batch.to(device)

        optimizer.zero_grad()
        outputs = model(X_batch)
        loss = criterion(outputs, y_batch)
        loss.backward()
        optimizer.step()

        train_losses.append(loss.item())

    # Validation
    model.eval()
    val_losses = []
    val_preds = []
    val_targets = []

    with torch.no_grad():
        for X_batch, y_batch in val_loader:
            X_batch, y_batch = X_batch.to(device), y_batch.to(device)
            outputs = model(X_batch)

            loss = criterion(outputs, y_batch)
            val_losses.append(loss.item())

            val_preds.append(outputs.cpu().numpy())
            val_targets.append(y_batch.cpu().numpy())

    val_preds = np.vstack(val_preds)
    val_targets = np.vstack(val_targets)
    mape_score = mean_absolute_percentage_error(val_targets, val_preds)

    print(f"Epoch {epoch+1}/{epochs} - Train Loss: {np.mean(train_losses):.5f} - Val MAPE: {mape_score:.5f}")

    if mape_score < best_val_loss:
        best_val_loss = mape_score
        torch.save(model.state_dict(), "best_model.pth")
        early_stop_counter = 0
    else:
        early_stop_counter += 1
        if early_stop_counter >= patience:
            print("Early stopping triggered.")
            break

# ---------------- Final Evaluation ---------------- #
model.load_state_dict(torch.load("best_model.pth"))
model.eval()

val_preds = []
with torch.no_grad():
    for X_batch, _ in val_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        val_preds.append(outputs.cpu().numpy())

val_preds = np.vstack(val_preds)
final_mape = mean_absolute_percentage_error(y_val, val_preds)
print(f"\nFinal Validation MAPE: {final_mape:.5f}")

# ---------------- Test Predictions & Submission ---------------- #
test_preds = []
with torch.no_grad():
    for X_batch in test_loader:
        X_batch = X_batch.to(device)
        outputs = model(X_batch)
        test_preds.append(outputs.cpu().numpy())

test_preds = np.vstack(test_preds)
submission = sample_submission.copy()
submission.iloc[:, 1:] = test_preds
submission['ID'] = test_df.index + 1  # Assuming IDs are sequential starting from 1
submission.to_csv('submission_nnmodel.csv', index=False)

print("\nTest predictions saved to 'submission_nnmodel.csv'")


Epoch 1/500 - Train Loss: 0.66831 - Val MAPE: 3.78487
Epoch 2/500 - Train Loss: 0.45864 - Val MAPE: 3.53578
Epoch 3/500 - Train Loss: 0.39474 - Val MAPE: 4.77921
Epoch 4/500 - Train Loss: 0.36389 - Val MAPE: 2.12169
Epoch 5/500 - Train Loss: 0.34585 - Val MAPE: 2.95270
Epoch 6/500 - Train Loss: 0.33665 - Val MAPE: 2.21883
Epoch 7/500 - Train Loss: 0.33443 - Val MAPE: 4.63032
Epoch 8/500 - Train Loss: 0.31593 - Val MAPE: 1.90787
Epoch 9/500 - Train Loss: 0.30925 - Val MAPE: 2.88519
Epoch 10/500 - Train Loss: 0.30722 - Val MAPE: 2.65295
Epoch 11/500 - Train Loss: 0.30515 - Val MAPE: 3.28633
Epoch 12/500 - Train Loss: 0.29959 - Val MAPE: 3.31348
Epoch 13/500 - Train Loss: 0.29776 - Val MAPE: 2.10139
Epoch 14/500 - Train Loss: 0.28462 - Val MAPE: 1.74815
Epoch 15/500 - Train Loss: 0.28011 - Val MAPE: 2.89409
Epoch 16/500 - Train Loss: 0.28475 - Val MAPE: 3.60577
Epoch 17/500 - Train Loss: 0.27985 - Val MAPE: 2.41472
Epoch 18/500 - Train Loss: 0.27014 - Val MAPE: 3.62665
Epoch 19/500 - Trai

In [ ]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_percentage_error
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, Dense, Dropout, BatchNormalization
from tensorflow.keras.callbacks import EarlyStopping, ReduceLROnPlateau
import lightgbm as lgb
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import Ridge

# --------------------------
# 1. Feature Engineering
# --------------------------
def create_features(df):
    """Generate advanced blend interaction features"""
    df = df.copy()
    
    # Component fractions
    fractions = [f'Component{i}_fraction' for i in range(1, 6)]
    
    # Create weighted properties
    for prop in range(1, 11):
        prop_cols = [f'Component{i}_Property{prop}' for i in range(1, 6)]
        df[f'Weighted_Property{prop}'] = sum(df[frac] * df[prop] for frac, prop in zip(fractions, prop_cols))
        
        # Non-linear interactions
        df[f'Max_Property{prop}'] = df[prop_cols].max(axis=1)
        df[f'Min_Property{prop}'] = df[prop_cols].min(axis=1)
        df[f'Range_Property{prop}'] = df[f'Max_Property{prop}'] - df[f'Min_Property{prop}']
    
    # Component interactions
    for i in range(1, 6):
        for j in range(i+1, 6):
            df[f'Frac_Interaction_{i}_{j}'] = df[f'Component{i}_fraction'] * df[f'Component{j}_fraction']
    
    # Property variance features
    for prop in range(1, 11):
        prop_cols = [f'Component{i}_Property{prop}' for i in range(1, 6)]
        df[f'Std_Property{prop}'] = df[prop_cols].std(axis=1)
        df[f'Median_Property{prop}'] = df[prop_cols].median(axis=1)
    
    return df

# --------------------------
# 2. Data Preparation
# --------------------------
# Load data
train = pd.read_csv('dataset/train.csv')
test = pd.read_csv('dataset/test.csv')

# Feature engineering
train_eng = create_features(train)
test_eng = create_features(test)

# Define features and targets
target_cols = [f'BlendProperty{i}' for i in range(1, 11)]
features = [col for col in train_eng.columns if col not in target_cols]

X = train_eng[features]
y = train_eng[target_cols]
X_test = test_eng[features]

# Train-validation split
X_train, X_val, y_train, y_val = train_test_split(
    X, y, test_size=0.2, random_state=42
)

# Feature scaling
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_val_scaled = scaler.transform(X_val)
X_test_scaled = scaler.transform(X_test)

# --------------------------
# 3. Neural Network Model
# --------------------------
def create_nn_model(input_dim, output_dim):
    inputs = Input(shape=(input_dim,))
    
    # Feature extraction branch
    x = Dense(512, activation='swish')(inputs)
    x = BatchNormalization()(x)
    x = Dropout(0.3)(x)
    
    x = Dense(256, activation='swish')(x)
    x = BatchNormalization()(x)
    x = Dropout(0.2)(x)
    
    # Property-specific branches
    branches = []
    for _ in range(output_dim):
        branch = Dense(128, activation='swish')(x)
        branch = Dense(64, activation='swish')(branch)
        branch = Dense(1)(branch)
        branches.append(branch)
    
    outputs = tf.keras.layers.concatenate(branches)
    model = Model(inputs=inputs, outputs=outputs)
    
    model.compile(
        optimizer=tf.keras.optimizers.Adam(learning_rate=0.001),
        loss='huber_loss',
        metrics=['mape']
    )
    return model

# Initialize model
nn_model = create_nn_model(X_train_scaled.shape[1], len(target_cols))

# Callbacks
callbacks = [
    EarlyStopping(patience=20, restore_best_weights=True),
    ReduceLROnPlateau(factor=0.5, patience=5)
]

# Train neural network
history = nn_model.fit(
    X_train_scaled, y_train.values,
    validation_data=(X_val_scaled, y_val.values),
    epochs=200,
    batch_size=64,
    callbacks=callbacks,
    verbose=1
)

# --------------------------
# 4. Gradient Boosting Model
# --------------------------
# Train LightGBM models for each target
lgb_models = {}
val_preds = pd.DataFrame(np.zeros(y_val.shape), columns=target_cols)
test_preds = pd.DataFrame(np.zeros((X_test.shape[0], len(target_cols))), 
                            columns=target_cols)

for i, target in enumerate(target_cols):
    print(f'\nTraining LightGBM for {target}')
    params = {
        'objective': 'mape',
        'boosting_type': 'gbdt',
        'num_leaves': 127,
        'learning_rate': 0.05,
        'feature_fraction': 0.8,
        'bagging_fraction': 0.8,
        'bagging_freq': 5,
        'verbose': -1,
        'n_jobs': -1,
        'lambda_l1': 0.5,
        'lambda_l2': 0.5
    }
    
    train_data = lgb.Dataset(X_train, label=y_train[target])
    val_data = lgb.Dataset(X_val, label=y_val[target])
    
    model = lgb.train(
        params,
        train_data,
        num_boost_round=2000,
        valid_sets=[val_data],
        callbacks=[
            lgb.early_stopping(stopping_rounds=100, verbose=True),
            lgb.log_evaluation(period=100)
        ]
    )
    
    lgb_models[target] = model
    val_preds[target] = model.predict(X_val)
    test_preds[target] = model.predict(X_test)

# --------------------------
# 5. Ensemble Modeling
# --------------------------
# Neural network predictions
nn_val_preds = nn_model.predict(X_val_scaled)
nn_test_preds = nn_model.predict(X_test_scaled)

# Create ensemble inputs
ensemble_val = pd.concat([
    pd.DataFrame(nn_val_preds, columns=target_cols).add_prefix('nn_'),
    val_preds.add_prefix('lgb_')
], axis=1)

ensemble_test = pd.concat([
    pd.DataFrame(nn_test_preds, columns=target_cols).add_prefix('nn_'),
    test_preds.add_prefix('lgb_')
], axis=1)


2025-07-07 20:34:14.674664: I metal_plugin/src/device/metal_device.cc:1154] Metal device set to: Apple M1
2025-07-07 20:34:14.674815: I metal_plugin/src/device/metal_device.cc:296] systemMemory: 8.00 GB
2025-07-07 20:34:14.674829: I metal_plugin/src/device/metal_device.cc:313] maxCacheSize: 2.67 GB
2025-07-07 20:34:14.675523: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:303] Could not identify NUMA node of platform GPU ID 0, defaulting to 0. Your kernel may not have been built with NUMA support.
2025-07-07 20:34:14.675834: I tensorflow/core/common_runtime/pluggable_device/pluggable_device_factory.cc:269] Created TensorFlow device (/job:localhost/replica:0/task:0/device:GPU:0 with 0 MB memory) -> physical PluggableDevice (device: 0, name: METAL, pci bus id: <undefined>)


Epoch 1/200


2025-07-07 20:34:16.679709: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


23/25 [==========================>...] - ETA: 0s - loss: 0.3351 - mape: 425.8800

2025-07-07 20:34:21.511317: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


25/25 [==============================] - 6s 51ms/step - loss: 0.3240 - mape: 415.5568 - val_loss: 0.2839 - val_mape: 279.4155 - lr: 0.0010
Epoch 2/200
25/25 [==============================] - 1s 26ms/step - loss: 0.1620 - mape: 312.0016 - val_loss: 0.2329 - val_mape: 277.5422 - lr: 0.0010
Epoch 3/200
25/25 [==============================] - 1s 24ms/step - loss: 0.1201 - mape: 275.0297 - val_loss: 0.1920 - val_mape: 248.0602 - lr: 0.0010
Epoch 4/200
25/25 [==============================] - 1s 24ms/step - loss: 0.1015 - mape: 235.2692 - val_loss: 0.1606 - val_mape: 203.8972 - lr: 0.0010
Epoch 5/200
25/25 [==============================] - 1s 24ms/step - loss: 0.0846 - mape: 240.5250 - val_loss: 0.1325 - val_mape: 244.5081 - lr: 0.0010
Epoch 6/200
25/25 [==============================] - 1s 24ms/step - loss: 0.0771 - mape: 235.4648 - val_loss: 0.1115 - val_mape: 262.6510 - lr: 0.0010
Epoch 7/200
25/25 [==============================] - 1s 24ms/step - loss: 0.0692 - mape: 203.8759 - val_lo

2025-07-07 20:37:15.612207: I tensorflow/core/grappler/optimizers/custom_graph_optimizer_registry.cc:114] Plugin optimizer for device_type GPU is enabled.


16/16 [==============================] - 0s 15ms/step


TypeError: zeros() got an unexpected keyword argument 'columns'

In [3]:

# Train stacking regressor
stacking_models = {}
final_preds = pd.DataFrame(np.zeros((ensemble_test.shape[0], len(target_cols))), columns=target_cols)

for i, target in enumerate(target_cols):
    # Use Ridge regression as meta-model
    stacking_model = Ridge(alpha=1.0)
    stacking_model.fit(ensemble_val, y_val[target])
    
    stacking_models[target] = stacking_model
    final_preds[target] = stacking_model.predict(ensemble_test)

# --------------------------
# 6. Evaluation & Submission
# --------------------------
# Evaluate on validation
val_mape = mean_absolute_percentage_error(y_val, final_preds.iloc[:len(y_val)])
print(f'\nFinal Validation MAPE: {val_mape:.4f}')

# Prepare submission
submission = pd.DataFrame({
    'ID': test['ID'],
    **{f'BlendProperty{i}': final_preds[f'BlendProperty{i}'] for i in range(1, 11)}
})

# Ensure correct column order
submission = submission[['ID'] + [f'BlendProperty{i}' for i in range(1, 11)]]

# Save predictions
submission.to_csv('final_submission.csv', index=False)


Final Validation MAPE: 10.0503
